# Sales Anomaly Analysis — Modeling

**Phase 1.** Isolation Forest (비지도)  
**Phase 2.** Random Forest + SMOTE (지도)  
**Phase 3.** 두 모델 비교

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['figure.dpi'] = 100

## 1. 데이터 로딩

In [ ]:
df = pd.read_csv('../data/df_final.csv')
print(f"shape: {df.shape}")
print(f"이상치 비율: {df['Is_Outlier'].mean()*100:.2f}% ({df['Is_Outlier'].sum()}건)")
display(df.head())

In [ ]:
# 피처 / 타깃 분리
feature_cols = [col for col in df.columns if col != 'Is_Outlier']
X = df[feature_cols]
y = df['Is_Outlier'].astype(int)
print(f"피처 수: {len(feature_cols)}")
print(feature_cols)

---
## Phase 1. Isolation Forest (비지도)

**목적:** 라벨 없이 14개 피처를 동시에 고려하는 다변량 이상치 탐지  
**contamination:** IQR 이상치 비율(75/2823)을 기준으로 설정

### 1.1 모델 학습

In [ ]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=75/2823,
    random_state=42
)

df['IF_Outlier'] = iso_forest.fit_predict(X) == -1  # -1: 이상치, 1: 정상
print(f"Isolation Forest 탐지 이상치: {df['IF_Outlier'].sum()}건")

### 1.2 IQR 결과와 교차 분석

In [ ]:
cross = pd.crosstab(df['Is_Outlier'], df['IF_Outlier'],
                    rownames=['IQR'], colnames=['Isolation Forest'])
print(cross)
print()

both     = ((df['Is_Outlier']==True) & (df['IF_Outlier']==True)).sum()
iqr_only = ((df['Is_Outlier']==True) & (df['IF_Outlier']==False)).sum()
if_only  = ((df['Is_Outlier']==False) & (df['IF_Outlier']==True)).sum()
neither  = ((df['Is_Outlier']==False) & (df['IF_Outlier']==False)).sum()

print(f"두 방법 모두 이상치 (고신뢰):  {both}건")
print(f"IQR만 이상치:                  {iqr_only}건  → 매출 고액, 다변량으론 정상")
print(f"Isolation Forest만 이상치:     {if_only}건  → IQR이 놓친 다변량 패턴")
print(f"둘 다 정상:                    {neither}건")

### 1.3 탐지 패턴 시각화

In [ ]:
def classify(row):
    if row['Is_Outlier'] and row['IF_Outlier']: return 'Both'
    elif row['Is_Outlier']: return 'IQR Only'
    elif row['IF_Outlier']: return 'IF Only'
    return 'Normal'

df['Outlier_Type'] = df.apply(classify, axis=1)
palette = {'Normal':'#95a5a6','IQR Only':'#378ADD','IF Only':'#e67e22','Both':'#E24B4A'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.scatterplot(data=df, x='QUANTITYORDERED', y='SALES',
                hue='Outlier_Type', palette=palette, alpha=0.6, s=50, ax=axes[0])
axes[0].set_title('SALES vs QUANTITYORDERED', fontweight='bold')

sns.scatterplot(data=df, x='PRICE_RATIO', y='SALES',
                hue='Outlier_Type', palette=palette, alpha=0.6, s=50, ax=axes[1])
axes[1].set_title('SALES vs PRICE_RATIO', fontweight='bold')

type_counts = df['Outlier_Type'].value_counts()
axes[2].bar(type_counts.index, type_counts.values,
            color=[palette[k] for k in type_counts.index], edgecolor='white')
axes[2].set_title('Outlier Detection Comparison', fontweight='bold')
for i, v in enumerate(type_counts.values):
    axes[2].text(i, v + 5, str(v), ha='center')

plt.tight_layout()
plt.savefig('../figures/fig_if_comparison.png', bbox_inches='tight')
plt.show()

### 1.4 IF만 탐지한 58건 특성 분석

In [ ]:
if_only_df = df[(df['Is_Outlier']==False) & (df['IF_Outlier']==True)]
print(f"IF Only 이상치: {len(if_only_df)}건")
display(if_only_df[['SALES','QUANTITYORDERED','PRICE_RATIO','BULK_ORDER_RATIO','REVENUE_CONTRIBUTION']].describe())

### 1.5 SHAP 변수 중요도 해석

In [ ]:
explainer = shap.TreeExplainer(iso_forest)
shap_values = explainer.shap_values(X)

# 전체 변수 중요도
plt.figure()
shap.summary_plot(shap_values, X, plot_type='bar', show=False)
plt.title('Feature Importance (Isolation Forest)')
plt.tight_layout()
plt.savefig('../figures/fig_shap_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 이상치 거래의 SHAP 방향
outlier_idx = df[df['IF_Outlier']==True].index
X_outliers = X.loc[outlier_idx]
shap_outliers = shap_values[outlier_idx]

plt.figure()
shap.summary_plot(shap_outliers, X_outliers, show=False)
plt.title('SHAP Values for Isolation Forest Outliers')
plt.tight_layout()
plt.savefig('../figures/fig_shap_outliers.png', bbox_inches='tight')
plt.show()

---
## Phase 2. Random Forest + SMOTE (지도)

**클래스 불균형:** 정상 2,748건 vs 이상치 75건 (36:1)  
**SMOTE:** 이상치 샘플 합성 보간으로 불균형 해소  
**파이프라인:** Cross Validation 각 fold에서 train에만 SMOTE 적용 → 데이터 누수 방지

### 2.1 데이터 분할

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  이상치: {y_train.sum()}건")
print(f"Test:  {X_test.shape}   이상치: {y_test.sum()}건")

### 2.2 SMOTE + Random Forest 파이프라인

In [ ]:
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

### 2.3 Cross Validation (StratifiedKFold, n=5)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                        scoring=['precision', 'recall', 'f1'],
                        return_train_score=False)

print("=== Cross Validation 결과 (5-Fold) ===")
print(f"Precision: {scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}")
print(f"Recall:    {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")
print(f"F1:        {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")

### 2.4 최종 테스트셋 평가

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Outlier']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Outlier'],
            yticklabels=['Normal', 'Outlier'])
plt.title('Confusion Matrix — Random Forest + SMOTE', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../figures/fig_rf_confusion.png', bbox_inches='tight')
plt.show()

### 2.5 Feature Importance

In [ ]:
rf_model = pipeline.named_steps['rf']
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='#378ADD')
plt.title('Feature Importance — Random Forest', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../figures/fig_rf_importance.png', bbox_inches='tight')
plt.show()

---
## Phase 3. 두 모델 최종 비교

### 3.1 탐지 결과 비교

In [ ]:
df['RF_Outlier'] = pipeline.predict(X).astype(bool)

rf_both     = ((df['Is_Outlier']==True) & (df['RF_Outlier']==True)).sum()
rf_iqr_only = ((df['Is_Outlier']==True) & (df['RF_Outlier']==False)).sum()
rf_if_only  = ((df['Is_Outlier']==False) & (df['RF_Outlier']==True)).sum()

print(f"{'':30} {'IQR':>8} {'IF':>8} {'RF':>8}")
print(f"{'총 탐지':30} {'75':>8} {'75':>8} {df['RF_Outlier'].sum():>8}")
print(f"{'IQR 일치':30} {'-':>8} {both:>8} {rf_both:>8}")
print(f"{'새로 발견 (IQR 미탐지)':30} {'-':>8} {if_only:>8} {rf_if_only:>8}")
print(f"{'IQR만 이상치':30} {'-':>8} {iqr_only:>8} {rf_iqr_only:>8}")

### 3.2 탐지 패턴 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

methods = ['IQR', 'Isolation Forest', 'Random Forest']
counts  = [75, 75, int(df['RF_Outlier'].sum())]
colors  = ['#378ADD', '#e67e22', '#1D9E75']
axes[0].bar(methods, counts, color=colors, edgecolor='white')
axes[0].set_title('총 탐지 건수 비교', fontweight='bold')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

labels = ['IQR 일치', 'IQR 미탐지\n(새 발견)']
if_vals = [both, if_only]
rf_vals = [rf_both, rf_if_only]
x = range(len(labels))
width = 0.35
axes[1].bar([i-width/2 for i in x], if_vals, width, label='Isolation Forest', color='#e67e22')
axes[1].bar([i+width/2 for i in x], rf_vals, width, label='Random Forest', color='#1D9E75')
axes[1].set_title('IQR 대비 탐지 패턴 비교', fontweight='bold')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels)
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/fig_model_comparison.png', bbox_inches='tight')
plt.show()

### 3.3 방법론 특성 요약

In [ ]:
summary = pd.DataFrame({
    '항목': ['학습 방식', '라벨 필요 여부', '핵심 판단 변수', '총 탐지', 'IQR 일치', '새로 발견', 'Recall'],
    'IQR': ['규칙 기반', '불필요', 'SALES (단일)', '75건', '-', '-', '-'],
    'Isolation Forest': ['비지도', '불필요', 'REVENUE_CONTRIBUTION, PRICE_RATIO',
                         '75건', f'{both}건', f'{if_only}건', '-'],
    'Random Forest': ['지도', '필요 (Is_Outlier)', 'SALES, BULK_ORDER_RATIO',
                      f"{df['RF_Outlier'].sum()}건", f'{rf_both}건', f'{rf_if_only}건', '0.883']
})
display(summary)